# Smart-Meter Energy Load Forecasting and Anomaly Detection

**Author:** Md. Fouad Iqbal  
**Background:** EEE graduate, ML learner  
**Goal:** Forecast short-term energy load and flag unusual meter behaviour.

In [ ]:
print("Smart-meter ML project workspace is ready.")

## 1. Data Loading and First Inspection

In [ ]:
# Kaggle attaches this dataset under /kaggle/input/
file_path = "/kaggle/input/smart-meter-electricity-consumption-dataset/smart_meter_data.csv"
df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and missing values:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
print("Summary statistics:")
display(df.describe(include="all").T)

In [ ]:
df.describe

### Initial observations

- Number of rows: 5000
- Number of columns: 7
- Target for forecasting: `Electricity Consumed (kWh)`
- Time column: `Timestamp`
- Available anomaly-label column: 0
- Columns with missing values: None

In [ ]:
import matplotlib.pyplot as plt

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.sort_values("Timestamp").reset_index(drop=True)

print("Invalid timestamps:", df["Timestamp"].isna().sum())
print("First timestamp:", df["Timestamp"].min())
print("Last timestamp:", df["Timestamp"].max())
print("Duplicate timestamps:", df["Timestamp"].duplicated().sum())

display(df[["Timestamp", "Electricity_Consumed"]].head())

In [ ]:
import seaborn as sns

plt.figure(figsize=(15, 6))
sns.lineplot(x='Timestamp', y='Electricity_Consumed', data=df)
plt.title('Electricity Consumed Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Electricity Consumed (kWh)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(
    df["Timestamp"],
    df["Electricity_Consumed"],
    color="steelblue",
    linewidth=0.8
)

plt.title("Smart-Meter Electricity Consumption Over Time")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.grid(alpha=0.3)
plt.show()

### Time-series validation

- Timestamp range: 2024-01-01 00:00:00 to 2024-04-14 03:30:00
- Invalid timestamps: 0
- Duplicate timestamps: 0
- What I notice in the energy plot: The plot shows fluctuations in electricity consumption over time, with no obvious long-term trend but some possible daily/weekly patterns and occasional spikes.

In [ ]:
time_diff = df["Timestamp"].diff().dropna()

print("Most common time intervals:")
print(time_diff.value_counts().head(10))

print("\nExpected readings per day for a 30-minute meter: 48")

In [ ]:
energy_col = "Electricity_Consumed"

print("Negative energy readings:", (df[energy_col] < 0).sum())
print("Zero energy readings:", (df[energy_col] == 0).sum())

display(df[df[energy_col] < 0][["Timestamp", energy_col]].head())

In [ ]:
df = df.dropna(subset=["Timestamp"]).copy()

print("Rows after removing invalid timestamps:", len(df))

### Data-quality decision

- Most common sampling interval: 0 days 00:30:00
- Negative-energy readings found: 0
- Cleaning performed: Removed only rows with invalid timestamps.
- Reason: Unusual energy values may be genuine anomalies and will be investigated later.

Electric load often changes by:
- Hour — morning/evening peaks;
- Day of week — weekday versus weekend routines;
- Month — seasonal effects.

In [ ]:
df["hour"] = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek
df["month"] = df["Timestamp"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["day_name"] = df["Timestamp"].dt.day_name()

display(
    df[
        [
            "Timestamp",
            "Electricity_Consumed",
            "hour",
            "day_of_week",
            "day_name",
            "month",
            "is_weekend",
        ]
    ].head()
)

In [ ]:
hourly_load = df.groupby("hour")["Electricity_Consumed"].mean()

plt.figure(figsize=(10, 4))
plt.plot(hourly_load.index, hourly_load.values, marker="o", color="darkorange")
plt.title("Average Electricity Consumption by Hour of Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Average Electricity Consumed (kWh)")
plt.xticks(range(24))
plt.grid(alpha=0.3)
plt.show()

The plot indicates slight peaks in electricity consumption around midday (11-12) and in the evening (18-20), which generally aligns with typical household or building demand patterns.

In [ ]:
daily_type_load = df.groupby('is_weekend')['Electricity_Consumed'].mean()

plt.figure(figsize=(7, 5))
sns.barplot(x=daily_type_load.index, y=daily_type_load.values, palette='viridis')
plt.title('Average Electricity Consumption: Weekday vs. Weekend')
plt.xlabel('Day Type (0=Weekday, 1=Weekend)')
plt.ylabel('Average Electricity Consumed (kWh)')
plt.xticks([0, 1], ['Weekday', 'Weekend'])
plt.grid(alpha=0.3)
plt.show()

The plot illustrates the difference in average electricity consumption between weekdays and weekends. We can observe whether consumption tends to be higher or lower on weekends compared to weekdays, which might reflect different usage patterns in a household or commercial setting.

In [ ]:
correlation = df['Temperature'].corr(df['Electricity_Consumed'])
print(f"Correlation between Temperature and Electricity_Consumed: {correlation:.4f}")

In [ ]:
humidity_correlation = df['Humidity'].corr(df['Electricity_Consumed'])
print(f"Correlation between Humidity and Electricity_Consumed: {humidity_correlation:.4f}")

Similar to temperature, this correlation value indicates the strength and direction of a linear relationship between humidity and electricity consumption. A value close to 0 suggests a weak or no linear relationship, while values closer to 1 or -1 suggest stronger positive or negative linear relationships, respectively. This will help us understand if humidity is a significant factor in electricity usage.

In [ ]:
numerical_cols = df.select_dtypes(include=['number']).columns
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

This heatmap visually represents the Pearson correlation coefficients between all pairs of numerical features. The color intensity and direction (red for positive, blue for negative) indicate the strength and type of correlation. Positive values indicate that as one variable increases, the other tends to increase, while negative values indicate that as one increases, the other tends to decrease. Values close to zero suggest a weak or no linear relationship. This visualization helps in quickly identifying strong relationships and potential multicollinearity among features.

The correlation coefficient indicates the strength and direction of a linear relationship between two variables. A value close to 1 suggests a strong positive linear relationship, a value close to -1 suggests a strong negative linear relationship, and a value close to 0 suggests a weak or no linear relationship. Based on the calculated value, we can infer how temperature changes might be associated with changes in electricity consumption.

In [ ]:
energy_col = "Electricity_Consumed"

df["lag_1"] = df[energy_col].shift(1)
df["lag_2"] = df[energy_col].shift(2)
df["lag_48"] = df[energy_col].shift(48)
df["lag_336"] = df[energy_col].shift(336)

display(
    df[
        [
            "Timestamp",
            energy_col,
            "lag_1",
            "lag_2",
            "lag_48",
            "lag_336",
        ]
    ].head(10)
)

In [ ]:
lag_columns = ["lag_1", "lag_2", "lag_48", "lag_336"]

print("Missing values created by lag features:")
print(df[lag_columns].isna().sum())

### Lag-feature interpretation

- `lag_1` represents consumption 30 minutes earlier.
- `lag_48` represents consumption at the same time on the previous day.
- The missing values at the beginning are expected because no prior readings exist there.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Electricity_Consumed'], kde=True, bins=50)
plt.title('Distribution of Electricity Consumed')
plt.xlabel('Electricity Consumed')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_model = df.dropna(subset=lag_columns).copy()

split_index = int(len(df_model) * 0.80)

train_df = df_model.iloc[:split_index].copy()
test_df = df_model.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df["Timestamp"].min(), "to", train_df["Timestamp"].max())

print("\nTesting period:")
print(test_df["Timestamp"].min(), "to", test_df["Timestamp"].max())

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    train_df["Timestamp"],
    train_df[energy_col],
    label="Training data",
    color="steelblue",
    linewidth=0.7,
)

plt.plot(
    test_df["Timestamp"],
    test_df[energy_col],
    label="Future test data",
    color="crimson",
    linewidth=0.7,
)

plt.title("Chronological Train/Test Split")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Evaluation design

I trained on the earlier 80% of readings and reserved the later 20% as unseen future data. This simulates how a deployed smart-meter system would forecast future load.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_test = test_df[energy_col]
baseline_pred = test_df["lag_1"]

mae_baseline = mean_absolute_error(y_test, baseline_pred)
rmse_baseline = np.sqrt(mean_squared_error(y_test, baseline_pred))

print(f"Persistence Baseline MAE:  {mae_baseline:.4f} kWh")
print(f"Persistence Baseline RMSE: {rmse_baseline:.4f} kWh")

In [ ]:
n_points = 200

plt.figure(figsize=(14, 5))
plt.plot(
    test_df["Timestamp"].iloc[:n_points],
    y_test.iloc[:n_points],
    label="Actual consumption",
    color="black",
    linewidth=1.5,
)
plt.plot(
    test_df["Timestamp"].iloc[:n_points],
    baseline_pred.iloc[:n_points],
    label="Persistence baseline",
    color="darkorange",
    linestyle="--",
    linewidth=1.2,
)

plt.title("Baseline Forecast vs Actual Consumption")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Baseline result

- Method: Persistence baseline (`lag_1`)
- MAE: 0.1882 kWh
- RMSE: 0.2322 kWh
- Interpretation: Every later ML model must beat this baseline on the same future test set.

In [ ]:
from sklearn.linear_model import LinearRegression

feature_columns = [
    "lag_1",
    "lag_2",
    "lag_48",
    "lag_336",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

X_train = train_df[feature_columns]
y_train = train_df[energy_col]

X_test = test_df[feature_columns]
y_test = test_df[energy_col]

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

mae_linear = mean_absolute_error(y_test, linear_pred)
rmse_linear = np.sqrt(mean_squared_error(y_test, linear_pred))

print(f"Linear Regression MAE:  {mae_linear:.4f} kWh")
print(f"Linear Regression RMSE: {rmse_linear:.4f} kWh")

In [ ]:
results = pd.DataFrame(
    {
        "Model": ["Persistence Baseline", "Linear Regression"],
        "MAE (kWh)": [mae_baseline, mae_linear],
        "RMSE (kWh)": [rmse_baseline, rmse_linear],
    }
)

display(results)

### First ML model result

- Model: Linear Regression
- Features: Four past-load values plus calendar features
- Does it beat the persistence baseline?: Yes, both MAE and RMSE are lower.
- Interpretation: The Linear Regression model, incorporating past load values and calendar features, significantly outperforms the simple persistence baseline. This indicates that these features provide valuable information for predicting electricity consumption. The model captures more complex patterns than just the previous time step's value.

In [ ]:
n_points = 200

plt.figure(figsize=(14, 5))

plt.plot(
    test_df["Timestamp"].iloc[:n_points],
    y_test.iloc[:n_points],
    label="Actual consumption",
    color="black",
    linewidth=1.5,
)

plt.plot(
    test_df["Timestamp"].iloc[:n_points],
    linear_pred[:n_points],
    label="Linear Regression forecast",
    color="forestgreen",
    linestyle="--",
    linewidth=1.2,
)

plt.title("Linear Regression Forecast vs Actual Consumption")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
residuals = y_test - linear_pred

plt.figure(figsize=(14, 4))
plt.plot(
    test_df["Timestamp"],
    residuals,
    color="purple",
    linewidth=0.7,
)
plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.title("Forecast Residuals Over the Test Period")
plt.xlabel("Time")
plt.ylabel("Actual − Predicted (kWh)")
plt.grid(alpha=0.3)
plt.show()

### Error-pattern observations

- Does the forecast follow the overall pattern?: Yes, the linear regression forecast generally follows the overall pattern of electricity consumption, but with some deviations.
- Are large errors isolated or repeated?: Both. There are some large, isolated errors, but there also seem to be recurring patterns of larger residuals, particularly during certain periods, suggesting systematic under- or over-prediction.
- Does the model more often overpredict or underpredict?: The residuals plot shows periods of both overprediction (negative residuals) and underprediction (positive residuals), fluctuating around zero. There are noticeable spikes indicating strong underprediction.
- Large residuals may later be treated as model-based anomaly candidates.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"Random Forest MAE:  {mae_rf:.4f} kWh")
print(f"Random Forest RMSE: {rmse_rf:.4f} kWh")

In [ ]:
results = pd.DataFrame(
    {
        "Model": [
            "Persistence Baseline",
            "Linear Regression",
            "Random Forest",
        ],
        "MAE (kWh)": [
            mae_baseline,
            mae_linear,
            mae_rf,
        ],
        "RMSE (kWh)": [
            rmse_baseline,
            rmse_linear,
            rmse_rf,
        ],
    }
).sort_values("MAE (kWh)")

display(results)

### Random Forest result

- Does Random Forest beat the baseline?: Yes, both MAE (0.1328) and RMSE (0.1652) are lower than the baseline's (MAE: 0.1882, RMSE: 0.2322).
- Does it beat Linear Regression?: No, Linear Regression has slightly better MAE (0.1310) and RMSE (0.1625) compared to Random Forest (MAE: 0.1328, RMSE: 0.1652).
- Current best forecasting model: Linear Regression

In [ ]:
rf_residuals = y_test - rf_pred

plt.figure(figsize=(14, 4))
plt.plot(
    test_df["Timestamp"],
    rf_residuals,
    color="darkgreen",
    linewidth=0.7,
)
plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.title("Random Forest Forecast Residuals Over the Test Period")
plt.xlabel("Time")
plt.ylabel("Actual − Predicted (kWh)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    test_df["Timestamp"],
    residuals,
    label="Linear Regression Residuals",
    color="purple",
    linewidth=0.7,
)

plt.plot(
    test_df["Timestamp"],
    rf_residuals,
    label="Random Forest Residuals",
    color="darkgreen",
    linewidth=0.7,
)

plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.title("Comparison of Linear Regression and Random Forest Residuals")
plt.xlabel("Time")
plt.ylabel("Actual − Predicted (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
feature_importance = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Importance": rf_model.feature_importances_,
    }
).sort_values("Importance", ascending=True)

display(feature_importance.sort_values("Importance", ascending=False))

In [ ]:
plt.figure(figsize=(9, 5))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"],
    color="teal",
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.show()

### Feature-importance interpretation

- Most important feature: `lag_2` (electricity consumption 1 hour ago)
- Second-most important feature: `lag_1` (electricity consumption 30 minutes ago)
- Are recent, daily, or weekly historical readings most useful?: Recent historical readings (`lag_1`, `lag_2`) are the most useful, followed closely by weekly (`lag_336`) and daily (`lag_48`) historical readings. All historical lag features are highly important.
- Deployment implication: We may prioritise the most useful low-cost features for ESP32 implementation. Given the high importance of all lag features, including `lag_336`, storing and accessing weekly historical data could be beneficial, but recent data (`lag_1`, `lag_2`) is critical.

In [ ]:
calibration_split = int(len(train_df) * 0.80)

fit_df = train_df.iloc[:calibration_split].copy()
calibration_df = train_df.iloc[calibration_split:].copy()

X_fit = fit_df[feature_columns]
y_fit = fit_df[energy_col]

X_calibration = calibration_df[feature_columns]
y_calibration = calibration_df[energy_col]

rf_calibration_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_calibration_model.fit(X_fit, y_fit)

calibration_pred = rf_calibration_model.predict(X_calibration)
calibration_abs_error = np.abs(y_calibration - calibration_pred)

anomaly_threshold = np.quantile(calibration_abs_error, 0.99)

print("Calibration rows:", len(calibration_df))
print(f"99th-percentile anomaly threshold: {anomaly_threshold:.4f} kWh")

In [ ]:
plt.figure(figsize=(9, 4))

plt.hist(calibration_abs_error, bins=40, color="slateblue", edgecolor="white")
plt.axvline(
    anomaly_threshold,
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"99th percentile = {anomaly_threshold:.4f} kWh",
)

plt.title("Calibration Absolute Forecast Errors")
plt.xlabel("Absolute error (kWh)")
plt.ylabel("Number of readings")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

### Anomaly-threshold decision

- Calibration period: the latest 20% of the training data
- Detection rule: flag a reading when absolute forecast error exceeds the 99th-percentile calibration error
- Threshold: 0.3976 kWh

In [ ]:
test_results = test_df[["Timestamp", energy_col]].copy()

test_results["Forecast (kWh)"] = rf_pred
test_results["Absolute Error (kWh)"] = np.abs(
    test_results[energy_col] - test_results["Forecast (kWh)"]
)

test_results["Forecast Anomaly"] = (
    test_results["Absolute Error (kWh)"] > anomaly_threshold
)

print("Total test readings:", len(test_results))
print("Forecast anomalies detected:", test_results["Forecast Anomaly"].sum())

display(
    test_results[test_results["Forecast Anomaly"]]
    .sort_values("Absolute Error (kWh)", ascending=False)
    .head(10)
)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    test_results["Timestamp"],
    test_results[energy_col],
    label="Actual consumption",
    color="steelblue",
    linewidth=0.8,
)

anomalies = test_results[test_results["Forecast Anomaly"]]

plt.scatter(
    anomalies["Timestamp"],
    anomalies[energy_col],
    label="Forecast anomaly",
    color="crimson",
    s=35,
    zorder=3,
)

plt.title("Forecast-Based Anomalies in the Future Test Period")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Forecast-based anomaly detection

- Total test readings: 933
- Anomalies detected: 10
- Interpretation: These readings differed from the Random Forest forecast by more than the calibration threshold.
- Caution: A flag is an investigation signal, not proof of an abnormal event.

In [ ]:
label_col = "Anomaly_Label"

print(test_df[label_col].value_counts(dropna=False))

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)

label_text = test_df[label_col].astype(str).str.strip().str.lower()

# Treat common anomaly representations as positive (1).
true_anomaly = label_text.isin(
    ["anomaly", "abnormal", "1", "true", "yes"]
).astype(int)

predicted_anomaly = test_results["Forecast Anomaly"].astype(int)

precision = precision_score(true_anomaly, predicted_anomaly, zero_division=0)
recall = recall_score(true_anomaly, predicted_anomaly, zero_division=0)
f1 = f1_score(true_anomaly, predicted_anomaly, zero_division=0)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

In [ ]:
cm = confusion_matrix(true_anomaly, predicted_anomaly)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Anomaly"],
)

display_cm.plot(cmap="Blues")
plt.title("Forecast-Based Detection vs Dataset Labels")
plt.show()

### Anomaly-detection evaluation

- Precision: 0.900
- Recall: 0.173
- F1-score: 0.290
- Limitation: The provided labels are generated labels, not independently verified field events.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

anomaly_feature_columns = [
    energy_col,
    "lag_1",
    "lag_2",
    "lag_48",
    "lag_336",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

X_anomaly_train = train_df[anomaly_feature_columns]
X_anomaly_test = test_df[anomaly_feature_columns]

scaler = StandardScaler()

X_anomaly_train_scaled = scaler.fit_transform(X_anomaly_train)
X_anomaly_test_scaled = scaler.transform(X_anomaly_test)

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
)

isolation_model.fit(X_anomaly_train_scaled)

# Isolation Forest returns -1 for anomaly and 1 for normal.
isolation_predictions = isolation_model.predict(X_anomaly_test_scaled)

test_results["Isolation Forest Anomaly"] = (
    isolation_predictions == -1
)

print(
    "Isolation Forest anomalies detected:",
    test_results["Isolation Forest Anomaly"].sum()
)

### Isolation Forest configuration

- Training data: historical training period only
- Input features: current load, lag features, and calendar features
- Contamination setting: 1%
- Important: this is an unsupervised detector; it was not trained using anomaly labels.

In [ ]:
def anomaly_metrics(y_true, y_pred):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "Anomalies Flagged": int(y_pred.sum()),
    }

forecast_metrics = anomaly_metrics(
    true_anomaly,
    test_results["Forecast Anomaly"].astype(int),
)

isolation_metrics = anomaly_metrics(
    true_anomaly,
    test_results["Isolation Forest Anomaly"].astype(int),
)

anomaly_comparison = pd.DataFrame(
    [forecast_metrics, isolation_metrics],
    index=["Forecast Residual", "Isolation Forest"],
)

display(anomaly_comparison)

In [ ]:
test_results["Detector Agreement"] = (
    test_results["Forecast Anomaly"]
    & test_results["Isolation Forest Anomaly"]
)

print(
    "Flagged by both detectors:",
    test_results["Detector Agreement"].sum()
)

display(
    test_results[
        test_results["Detector Agreement"]
    ][
        [
            "Timestamp",
            energy_col,
            "Forecast (kWh)",
            "Absolute Error (kWh)",
            "Forecast Anomaly",
            "Isolation Forest Anomaly",
        ]
    ].head(10)
)

### Detector comparison

- Better F1-score: Forecast Residual (0.290) compared to Isolation Forest (0.094)
- Forecast-based anomalies: 10
- Isolation Forest anomalies: 12
- Agreement count: 1
- Interpretation: Events flagged by both methods are higher-priority investigation candidates.

In [ ]:
n_points = 500
plot_df = test_results.iloc[:n_points].copy()

forecast_only = plot_df[
    plot_df["Forecast Anomaly"]
    & ~plot_df["Isolation Forest Anomaly"]
]

isolation_only = plot_df[
    ~plot_df["Forecast Anomaly"]
    & plot_df["Isolation Forest Anomaly"]
]

both_detectors = plot_df[
    plot_df["Forecast Anomaly"]
    & plot_df["Isolation Forest Anomaly"]
]

plt.figure(figsize=(15, 6))

plt.plot(
    plot_df["Timestamp"],
    plot_df[energy_col],
    color="steelblue",
    linewidth=0.9,
    label="Actual consumption",
)

plt.scatter(
    forecast_only["Timestamp"],
    forecast_only[energy_col],
    color="darkorange",
    marker="x",
    s=55,
    label="Forecast only",
    zorder=3,
)

plt.scatter(
    isolation_only["Timestamp"],
    isolation_only[energy_col],
    color="purple",
    marker="^",
    s=45,
    label="Isolation Forest only",
    zorder=3,
)

plt.scatter(
    both_detectors["Timestamp"],
    both_detectors[energy_col],
    color="crimson",
    marker="o",
    s=65,
    label="Both detectors",
    zorder=4,
)

plt.title("Comparison of Smart-Meter Anomaly Detectors")
plt.xlabel("Time")
plt.ylabel("Electricity Consumed (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Visual detector comparison

- Plot period: first 500 future test readings
- Do the two methods flag the same events?: Mostly no, only 1 event was flagged by both.
- Which detector identifies more unusual patterns?: Isolation Forest identifies more (12) compared to Forecast-based (10).
- Higher-priority candidates: events flagged by both detectors.

In [ ]:
test_results["Ensemble OR"] = (
    test_results["Forecast Anomaly"]
    | test_results["Isolation Forest Anomaly"]
)

test_results["Ensemble AND"] = (
    test_results["Forecast Anomaly"]
    & test_results["Isolation Forest Anomaly"]
)

ensemble_or_metrics = anomaly_metrics(
    true_anomaly,
    test_results["Ensemble OR"].astype(int),
)

ensemble_and_metrics = anomaly_metrics(
    true_anomaly,
    test_results["Ensemble AND"].astype(int),
)

all_detector_results = pd.DataFrame(
    [
        forecast_metrics,
        isolation_metrics,
        ensemble_or_metrics,
        ensemble_and_metrics,
    ],
    index=[
        "Forecast Residual",
        "Isolation Forest",
        "Ensemble OR",
        "Ensemble AND",
    ],
)

display(all_detector_results)

### Ensemble decision

- Rule with highest recall: Ensemble OR
- Rule with highest precision: Ensemble AND
- Rule with highest F1-score: Ensemble OR
- Recommended prototype rule: Ensemble OR
- Reason for the choice: The Ensemble OR rule offers the highest F1-score, indicating the best balance between precision and recall among the evaluated methods. This makes it a robust choice for a prototype anomaly detection system, as it aims to minimize both false positives and false negatives effectively.

In [ ]:
alert_table = test_results[
    test_results["Ensemble OR"]
].copy()

alert_table["Severity"] = np.where(
    alert_table["Ensemble AND"],
    "High",
    "Medium",
)

alert_table["Triggered By"] = np.select(
    [
        alert_table["Ensemble AND"],
        alert_table["Forecast Anomaly"],
        alert_table["Isolation Forest Anomaly"],
    ],
    [
        "Forecast Residual + Isolation Forest",
        "Forecast Residual",
        "Isolation Forest",
    ],
    default="Unknown",
)

alert_table = alert_table[
    [
        "Timestamp",
        energy_col,
        "Forecast (kWh)",
        "Absolute Error (kWh)",
        "Severity",
        "Triggered By",
    ]
].sort_values(
    ["Severity", "Absolute Error (kWh)"],
    ascending=[True, False],
)

display(alert_table.head(15))

print("Total alerts:", len(alert_table))
print("\nAlerts by severity:")
print(alert_table["Severity"].value_counts())

### Alert-export design

- File: `smart_meter_anomaly_alerts.csv`
- High severity: both detectors agree
- Medium severity: one detector flags the reading
- Embedded direction: an ESP32 could publish only this compact alert information via Wi-Fi.